## Reading files from ADLSgen2

In [0]:
# dbutils.widgets.text("Table_Schema", "")
# dbutils.widgets.text("Table_Name", "")

# tableName = dbutils.widgets.get("Table_Name").lower()
# tableSchema = dbutils.widgets.get("Table_Schema").lower()

# def read_all_tables():
#     path = f"abfss://landing-layer@stadlsgen2eus0426.dfs.core.windows.net/OnpermSql_incremental_Loads/{tableSchema}/{tableSchema}_{tableName}/{tableName}.csv"
#     files = dbutils.fs.ls(path)
    
#     table_dfs = {}
#     for file in files:
#         if file.name.endswith('.csv'):
#             df = (spark.read.format("csv")
#                   .option("header", True)
#                   .load(file.path))
#             table_dfs[file.name] = df
#             df.show()   
            
#     return table_dfs
# read_all_tables()


## AutoLoader 

In [0]:

dbutils.widgets.text("Table_Schema", "")
dbutils.widgets.text("Table_Name", "")

tableName = dbutils.widgets.get("Table_Name").lower()
tableSchema = dbutils.widgets.get("Table_Schema").lower()

print(f"Processing: {tableSchema}.{tableName}")


spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")


if tableSchema == "netflix":
    catalog = "adb-netflix"
else:
    catalog = "adv2019"

schema = "bronze-layer"


target_table = f"`{catalog}`.`{schema}`.{tableName}"

print(f"Target table: {target_table}")

target_path = f"abfss://bronze-layer@stadlsgen2eus0426.dfs.core.windows.net/"

base_path = "abfss://landing-layer@stadlsgen2eus0426.dfs.core.windows.net/OnpermSql_incremental_Loads"

source_path = f"{base_path}/{tableSchema}/{tableSchema}_{tableName}/"
checkpoint_path = f"{target_path}/{catalog}/_checkpoints/{tableSchema}/{tableName}/"
schema_path = f"{target_path}/{catalog}/_schemas/{tableSchema}/{tableName}/"

print(f"Source path: {source_path}")


df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "rescue")  
    .option("header", "true")
    .load(source_path)
)

from pyspark.sql.functions import current_timestamp, input_file_name

df = (
    df.withColumn("ingestion_time", current_timestamp())
     
)

(
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")   
    .outputMode("append")
    .trigger(once=True) 
    .toTable(target_table)
)

In [0]:
# files = dbutils.fs.ls('abfss://landing-layer@stadlsgen2eus0426.dfs.core.windows.net/Netflix_files/')
# display(files)

In [0]:
# df_titles = spark.read.format('csv').option('header','true').load('abfss://landing-layer@stadlsgen2eus0426.dfs.core.windows.net/Netflix_files/netflix_titles.csv')
# # display(df_titles)
# df_cast = spark.read.format('csv').option('header','true').load('abfss://landing-layer@stadlsgen2eus0426.dfs.core.windows.net/Netflix_files/netflix_cast.csv')
# # display(df_cast)
# df_country = spark.read.format('csv').option('header','true').load('abfss://landing-layer@stadlsgen2eus0426.dfs.core.windows.net/Netflix_files/netflix_countries.csv')
# # display(df_country)
# df_director = spark.read.format('csv').option('header','true').load('abfss://landing-layer@stadlsgen2eus0426.dfs.core.windows.net/Netflix_files/netflix_directors.csv')
# display(df_director)

In [0]:
# df_titles.write.saveAsTable("`adb-netflix`.bronze-layer.netflix_titles_managed")
# df_cast.write.saveAsTable("`adb-netflix`.bronze-layer.netflix_cast_managed")
# df_country.write.saveAsTable("`adb-netflix`.bronze-layer.netflix_country_managed")
# df_director.write.saveAsTable("`adb-netflix`.bronze-layer.netflix_director_managed")

In [0]:
# from pyspark.sql.functions import col, current_timestamp

# # Step 1: Replace date_added with current timestamp for all rows
# df_titles = df.withColumn("date_added", current_timestamp())

# # Step 2: Select and cast columns properly
# df_titles = df_titles.select(
#     col("show_id").cast("bigint"),
#     col("title").cast("string"),
#     col("type").cast("string"),
#     col("release_year").cast("int"),
#     col("rating").cast("string"),
#     col("date_added").cast("timestamp"),
#     col("description").cast("string"),
#     col("duration_minutes").cast("int"),
#     col("duration_seasons").cast("int")
# )

# # Step 3: Directors dataframe
# df_directors = df_director.select(
#     col("director").cast("string"),
#     col("show_id").cast("bigint")
# )

# Step 4: Display results
# display(df_titles)
# display(df_directors)

In [0]:
# df_titles.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("`adb-netflix`.`bronze-layer`.`netflix_titles`")

# df_cast.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
#     "`adb-netflix`.`bronze-layer`.`netflix_cast`"
# )
# df_country.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
#     "`adb-netflix`.`bronze-layer`.`netflix_country`"
# )
# df_director.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
#     "`adb-netflix`.`bronze-layer`.`netflix_director`")
# # )

## Writing into bronze-Schema

In [0]:
# table_dfs = read_all_tables()
# df = list(table_dfs.values())[0]

# if tableSchema == "netflix":
#     catalog = "adb-netflix"
# else:
#     catalog = "adv2019"

# schema = "bronze-layer"


# target_table = f"`{catalog}`.`{schema}`.{tableName}"

# df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(target_table)